[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day3_Remote_Sensing_Fundamentals_and_Indices/04_digital_image_processing_in_gee_es.ipynb)

# Día 3 - Fundamentos de Teledetección e Índices Espectrales
## Sesión 4: Procesamiento digital de imágenes en GEE

| | |
|---|---|
| **Fecha** | 2026-10-14 (miércoles) |
| **Horario** | 15:40 - 17:10 |
| **Entidad** | UoM |
| **Instructores** | Christhian, Sebastian |

### Contenido

> Preparación de un **cubo de datos** de imágenes en Google Earth Engine: área de estudio, colección Landsat 8/9,
> filtrado por nubes, factor de escala, máscara de nubes, índices espectrales como nuevas bandas, estadísticas
> zonales sobre una región de interés, variabilidad del NDVI en el tiempo y máscaras por umbral.

### Objetivos de aprendizaje

- [ ] Construir una colección Landsat 8 + 9 limpia: filtrar, escalar y eliminar los píxeles con nubes
- [ ] Escribir los índices **NDVI, MSAVI, EVI, NDWI y NDMI** como funciones y agregarlos como bandas
- [ ] Extraer estadísticas zonales (mínimo, media, máximo, desviación estándar) para una región dibujada en el
      mapa y leerlas como un `DataFrame` y como un gráfico
- [ ] Entender que el valor de un píxel **cambia en el tiempo** (NDVI mínimo, máximo y amplitud)
- [ ] Crear máscaras simples por umbral para vegetación y agua

### Datos necesarios

No se usan archivos locales. `LANDSAT/LC08/C02/T1_L2` y `LANDSAT/LC09/C02/T1_L2` (Colección 2, reflectancia
de superficie de Nivel 2) del catálogo de Earth Engine.

### Por qué esto importa

Todo lo que se hace aquí - píxeles limpios, unidades físicas, bandas adicionales con índices - es la
preparación del **cubo de datos** (x, y, bandas, tiempo) que usaremos después para la **clasificación
de imágenes**. Un clasificador solo es tan bueno como las bandas que le damos.

---

## 4.0 - Configuración

In [ ]:
# Librerías (descomente la línea de pip en un entorno nuevo, p. ej. Google Colab)
#%pip install -q earthengine-api geemap

import ee                         # API de Google Earth Engine
import geemap                     # mapas interactivos para Earth Engine
import pandas as pd               # tablas (DataFrames)
import matplotlib.pyplot as plt   # gráficos

In [ ]:
# Autenticar e inicializar Earth Engine
# >>> REEMPLAZAR por el id de su propio proyecto de Cloud <<<
EE_PROJECT = "ee-christhianscunha"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

---

## 4.1 - Área de estudio y colección de imágenes

El área de estudio es una franja del altiplano boliviano que va desde la orilla del **Lago Menor**, a través
de la planicie de **Katari**, hasta el borde de **El Alto**. Contiene agua, cultivos, suelo desnudo y
áreas urbanas - superficies muy diferentes, lo que facilita comparar los índices.

### Landsat 8 y 9, Colección 2, Nivel 2

Ambos satélites llevan sensores casi idénticos (OLI y OLI-2), por lo que podemos **unirlos** en
una sola colección y obtener una imagen cada ~8 días en lugar de cada 16.

| Banda | Nombre que usaremos | Longitud de onda |
|---|---|---|
| SR_B2 | Blue | 0.45 - 0.51 um |
| SR_B3 | Green | 0.53 - 0.59 um |
| SR_B4 | Red | 0.64 - 0.67 um |
| SR_B5 | NIR | 0.85 - 0.88 um |
| SR_B6 | SWIR1 | 1.57 - 1.65 um |
| SR_B7 | SWIR2 | 2.11 - 2.29 um |

In [ ]:
# Área de estudio (AOI): un rectángulo [oeste, sur, este, norte]
AOI = ee.Geometry.Rectangle([-68.90, -16.62, -68.10, -16.30])

# Mostrar el AOI en un mapa
Map = geemap.Map()
Map.centerObject(AOI, 10)
Map.addLayer(AOI, {"color": "red"}, "AOI")
Map

In [ ]:
# Unir Landsat 8 y Landsat 9 y filtrar por área, fecha y cobertura de nubes
START, END = "2024-01-01", "2024-12-31"

landsat = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
             .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
             .filterBounds(AOI)                       # solo escenas que tocan el AOI
             .filterDate(START, END)                  # solo un año
             .filter(ee.Filter.lt("CLOUD_COVER", 30)) # escenas con menos de 30% de nubes
             .sort("system:time_start"))              # ordenar por fecha

print("Number of scenes:", landsat.size().getInfo())

In [ ]:
# Ver las escenas como tabla: fecha y cobertura de nubes de cada una
scenes = pd.DataFrame({
    "date": landsat.aggregate_array("DATE_ACQUIRED").getInfo(),
    "satellite": landsat.aggregate_array("SPACECRAFT_ID").getInfo(),
    "cloud_cover (%)": landsat.aggregate_array("CLOUD_COVER").getInfo(),
})
scenes.head(10)

### Factor de escala y máscara de nubes

**Factor de escala.** Landsat Nivel 2 almacena la reflectancia como números enteros para ahorrar espacio. Para obtener el
valor físico (reflectancia, aproximadamente de 0 a 1) aplicamos:

$$\rho = DN \times 0.0000275 - 0.2$$

Sin este paso, EVI y MSAVI dan valores incorrectos, y umbrales como `NDVI > 0.3` pierden
sentido al compararlos con otros estudios.

**Máscara de nubes.** El filtro a nivel de escena (`CLOUD_COVER < 30`) no es suficiente: una escena con 10%
de nubes todavía tiene píxeles nublados. La banda `QA_PIXEL` marca cada píxel, bit a bit:

| Bit | Significado |
|---|---|
| 1 | nube dilatada (borde de nube) |
| 2 | cirros |
| 3 | nube |
| 4 | sombra de nube |

Conservamos solo los píxeles donde **ninguno** de estos bits está activado, usando `updateMask`.

In [ ]:
def mask_clouds(image):
    # QA_PIXEL: bits 1 (nube dilatada), 2 (cirros), 3 (nube) y 4 (sombra)
    qa = image.select("QA_PIXEL")
    cloud_bits = (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4)
    clear = qa.bitwiseAnd(cloud_bits).eq(0)      # 1 = píxel despejado, 0 = nube o sombra
    return image.updateMask(clear)               # los píxeles con nubes pasan a ser "sin datos"


def scale_and_rename(image):
    # Aplicar el factor de escala de la Colección 2 a las bandas ópticas
    optical = image.select(["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"])
    optical = optical.multiply(0.0000275).add(-0.2)
    # Dar a las bandas nombres legibles
    optical = optical.rename(["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"])
    # Conservar las propiedades originales (fecha, cobertura de nubes...) de la imagen
    return ee.Image(optical.copyProperties(image, ["system:time_start", "CLOUD_COVER", "DATE_ACQUIRED"]))


# Aplicar ambas funciones a cada imagen de la colección
collection = landsat.map(mask_clouds).map(scale_and_rename)

print("Bands:", collection.first().bandNames().getInfo())

---

## 4.2 - Índices espectrales como nuevas bandas

Un índice es simplemente **otra banda**. Escribimos una función que calcula los cinco índices y
los añade a la imagen con `addBands`. Aplicada con `.map()`, cada imagen de la
colección recibe las nuevas bandas.

| Índice | Fórmula | Qué muestra |
|---|---|---|
| **NDVI** | (NIR - Red) / (NIR + Red) | vigor de la vegetación |
| **MSAVI** | (2 NIR + 1 - sqrt((2 NIR + 1)^2 - 8 (NIR - Red))) / 2 | vegetación, reduce la influencia del suelo (vegetación dispersa, como en el altiplano) |
| **EVI** | 2.5 (NIR - Red) / (NIR + 6 Red - 7.5 Blue + 1) | vegetación, menor saturación en doseles densos |
| **NDWI** | (Green - NIR) / (Green + NIR) | agua abierta (McFeeters, 1996) |
| **NDMI** | (NIR - SWIR1) / (NIR + SWIR1) | humedad en la vegetación y el suelo |

`normalizedDifference` cubre los cocientes simples. EVI y MSAVI necesitan `expression`, que nos permite
escribir la fórmula casi como en el papel.

In [ ]:
def add_indices(image):
    # Diferencias normalizadas: (A - B) / (A + B)
    ndvi = image.normalizedDifference(["NIR", "Red"]).rename("NDVI")
    ndwi = image.normalizedDifference(["Green", "NIR"]).rename("NDWI")
    ndmi = image.normalizedDifference(["NIR", "SWIR1"]).rename("NDMI")

    # EVI: fórmula escrita con expression()
    evi = image.expression(
        "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
        {"NIR": image.select("NIR"), "RED": image.select("Red"), "BLUE": image.select("Blue")},
    ).rename("EVI")

    # MSAVI: fórmula escrita con expression()
    msavi = image.expression(
        "(2 * NIR + 1 - sqrt((2 * NIR + 1) ** 2 - 8 * (NIR - RED))) / 2",
        {"NIR": image.select("NIR"), "RED": image.select("Red")},
    ).rename("MSAVI")

    # Agregar los cinco índices como nuevas bandas
    return image.addBands([ndvi, msavi, evi, ndwi, ndmi])


# Cada imagen de la colección tiene ahora 6 bandas ópticas + 5 índices
collection = collection.map(add_indices)

print("Bands:", collection.first().bandNames().getInfo())

Esta colección es nuestro **cubo de datos**: para cada fecha (tiempo) tenemos una imagen (x, y) con 11
bandas. Está limpia (sin nubes), en unidades físicas (reflectancia) y enriquecida con índices.

---

## 4.3 - Una escena y estadísticas zonales

Elegimos la **escena con menos nubes** del año. Una escena Landsat puede no cubrir todo el AOI,
así que tomamos la fecha de esa escena y hacemos un **mosaico** con todas las imágenes adquiridas ese mismo día
(escenas vecinas de la misma órbita).

In [ ]:
# La imagen con menos nubes y su fecha
best = collection.sort("CLOUD_COVER").first()
date = ee.Date(best.get("system:time_start"))

# Mosaico de todas las imágenes de ese día, recortado al AOI
scene = collection.filterDate(date, date.advance(1, "day")).mosaic().clip(AOI)

print("Scene date:", date.format("YYYY-MM-dd").getInfo(),
      "| cloud cover:", round(best.get("CLOUD_COVER").getInfo(), 1), "%")

### Dibuje su región de interés (ROI)

Ejecute la siguiente celda y use las **herramientas de dibujo** a la izquierda del mapa para dibujar un rectángulo o
un polígono (por ejemplo sobre una parcela de cultivo, el lago o la ciudad). Luego ejecute la celda siguiente.

Si no dibuja nada, se usa una ROI por defecto (un círculo de 500 m en la planicie de Katari).

In [ ]:
# Parámetros de visualización
vis_rgb = {"bands": ["Red", "Green", "Blue"], "min": 0.0, "max": 0.3}    # color natural
vis_false = {"bands": ["NIR", "Red", "Green"], "min": 0.0, "max": 0.4}   # falso color (vegetación en rojo)

# Un nuevo mapa para la escena
Map_scene = geemap.Map()
Map_scene.centerObject(AOI, 10)
Map_scene.addLayer(scene, vis_rgb, "RGB (natural colour)")
Map_scene.addLayer(scene, vis_false, "False colour (NIR-R-G)", False)
Map_scene

In [ ]:
# Tomar la geometría dibujada en el mapa (None si no se dibujó nada)
roi = Map_scene.user_roi

# ROI por defecto: un buffer de 500 m alrededor de un punto en la planicie de Katari
if roi is None:
    roi = ee.Geometry.Point([-68.55, -16.50]).buffer(500)

# Mostrar la ROI en el mapa
Map_scene.addLayer(roi, {"color": "yellow"}, "ROI")
Map_scene.centerObject(roi, 14)

### Estadísticas zonales con `reduceRegion`

Una ROI contiene muchos píxeles, y **no todos tienen el mismo valor**. `reduceRegion`
los resume. Combinando reductores obtenemos, en una sola llamada, el **mínimo, la media, el máximo y la desviación
estándar** de cada banda dentro de la ROI.

Lo escribimos una vez como una función que devuelve un `DataFrame`, y una segunda función que dibuja
el gráfico. Luego usamos ambas para las bandas ópticas, los índices de vegetación y los índices
de agua.

In [ ]:
def zonal_stats(image, bands, region, scale=30):
    # Un reductor que calcula media, mínimo, máximo y desviación estándar a la vez
    reducer = (ee.Reducer.mean()
               .combine(ee.Reducer.minMax(), sharedInputs=True)
               .combine(ee.Reducer.stdDev(), sharedInputs=True))

    # Resumir los píxeles de la región (30 m = tamaño del píxel Landsat)
    stats = image.select(bands).reduceRegion(
        reducer=reducer, geometry=region, scale=scale, maxPixels=1e9).getInfo()

    # Organizar el resultado: una fila por banda, una columna por estadística
    table = {s: [stats[f"{b}_{s}"] for b in bands] for s in ["min", "mean", "max", "stdDev"]}
    return pd.DataFrame(table, index=bands).round(3)


def plot_stats(df, title, ylabel, color):
    fig, ax = plt.subplots(figsize=(8, 4))
    # Barras: valor medio, con la desviación estándar como barra de error
    ax.bar(df.index, df["mean"], yerr=df["stdDev"], capsize=5,
           color=color, alpha=0.8, label="mean ± std. dev.")
    # Marcadores: valor mínimo y máximo de los píxeles en la ROI
    ax.scatter(df.index, df["min"], marker="v", color="black", zorder=3, label="min")
    ax.scatter(df.index, df["max"], marker="^", color="black", zorder=3, label="max")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Band")
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()

#### Bandas ópticas

Observe la forma de las barras de Blue a SWIR2: es la **firma espectral** de la
ROI. La vegetación tiene Red bajo y NIR alto; el agua es baja en todas las bandas y casi cero en NIR y
SWIR; el suelo desnudo aumenta gradualmente de Blue a SWIR.

In [ ]:
optical_bands = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]

df_optical = zonal_stats(scene, optical_bands, roi)
df_optical

In [ ]:
plot_stats(df_optical, "Optical bands in the ROI", "Surface reflectance", "#7f8c8d")

#### Índices de vegetación

In [ ]:
veg_indices = ["NDVI", "MSAVI", "EVI"]

df_veg = zonal_stats(scene, veg_indices, roi)
df_veg

In [ ]:
plot_stats(df_veg, "Vegetation indices in the ROI", "Index value", "#2e8b57")

#### Índices de agua y humedad

In [ ]:
water_indices = ["NDWI", "NDMI"]

df_water = zonal_stats(scene, water_indices, roi)
df_water

In [ ]:
plot_stats(df_water, "Water and moisture indices in the ROI", "Index value", "#2c7bb6")

**Cómo leer los gráficos.** La barra es la media, la barra de error la desviación estándar, y
los triángulos el mínimo y el máximo. Una gran distancia entre mínimo y máximo significa que la ROI **no es
homogénea** - mezcla superficies (una parcela y su borde, agua y orilla). Esta variabilidad
es justamente con lo que tiene que lidiar un clasificador.

Dibuje una nueva ROI sobre otra superficie, vuelva a ejecutar las celdas y compare.

### Lo mismo con `geemap.zonal_stats`

geemap ofrece una función lista que hace lo mismo con `reduceRegions`, para una
`FeatureCollection` (una o varias zonas), y guarda el resultado en CSV.

In [ ]:
# La ROI como FeatureCollection (geemap.zonal_stats trabaja con zonas/polígonos)
roi_fc = ee.FeatureCollection([ee.Feature(roi, {"zone": "ROI"})])

# Media de cada banda e índice en la ROI, guardada en un archivo CSV
geemap.zonal_stats(scene, roi_fc, "zonal_stats_roi.csv", stat_type="MEAN", scale=30)

pd.read_csv("zonal_stats_roi.csv").round(3)

---

## 4.4 - El NDVI cambia en el tiempo

Una sola escena es un momento. En el altiplano la vegetación crece en la época de lluvias
(diciembre-marzo) y se seca en invierno (junio-agosto). Reduciendo todo el **año** píxel por píxel
obtenemos:

- **NDVI mínimo** - el momento más seco de cada píxel
- **NDVI máximo** - el momento más verde de cada píxel
- **Amplitud** = máximo - mínimo - cuánto **cambia** cada píxel durante el año

El agua y las áreas urbanas cambian poco (amplitud baja). Los cultivos cambian mucho (amplitud alta).
Por eso el tiempo es una dimensión importante del cubo de datos para la clasificación.

In [ ]:
# Banda NDVI de cada imagen del año
ndvi_year = collection.select("NDVI")

# Mínimo, máximo y amplitud píxel por píxel
ndvi_min = ndvi_year.min().clip(AOI)
ndvi_max = ndvi_year.max().clip(AOI)
ndvi_amp = ndvi_max.subtract(ndvi_min).rename("NDVI_amplitude")

# Paletas de colores
vis_ndvi = {"min": -0.2, "max": 0.8, "palette": ["#a6611a", "#f5f5dc", "#a6d96a", "#1a9641", "#00441b"]}
vis_amp = {"min": 0.0, "max": 0.6, "palette": ["#ffffcc", "#fd8d3c", "#bd0026"]}

Map_ndvi = geemap.Map()
Map_ndvi.centerObject(AOI, 10)
Map_ndvi.addLayer(scene, vis_rgb, "RGB scene", False)
Map_ndvi.addLayer(ndvi_min, vis_ndvi, "NDVI min")
Map_ndvi.addLayer(ndvi_max, vis_ndvi, "NDVI max")
Map_ndvi.addLayer(ndvi_amp, vis_amp, "NDVI amplitude (max - min)")
Map_ndvi.addLayer(roi, {"color": "yellow"}, "ROI")
Map_ndvi.add_colorbar(vis_ndvi, label="NDVI (min / max)")
Map_ndvi.add_colorbar(vis_amp, label="NDVI amplitude")
Map_ndvi

Use el control de capas (arriba a la derecha) para cambiar entre **NDVI mínimo** y **NDVI máximo**: el mismo
píxel tiene valores muy diferentes según la fecha. La capa de amplitud resume esa
diferencia en una sola imagen.

### El NDVI en el tiempo dentro de la ROI

Para verlo en un gráfico calculamos el NDVI medio de la ROI para cada imagen de la colección.

In [ ]:
def ndvi_in_roi(image):
    # NDVI medio de la ROI para una imagen, guardado como propiedad de un Feature
    mean = image.select("NDVI").reduceRegion(ee.Reducer.mean(), roi, 30).get("NDVI")
    return ee.Feature(None, {"date": image.date().format("YYYY-MM-dd"), "NDVI": mean})


# Un Feature por imagen; descartar las fechas en que la ROI estaba cubierta por nubes
series = ee.FeatureCollection(collection.map(ndvi_in_roi)).filter(ee.Filter.notNull(["NDVI"]))

df_series = pd.DataFrame({
    "date": pd.to_datetime(series.aggregate_array("date").getInfo()),
    "NDVI": series.aggregate_array("NDVI").getInfo(),
})

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df_series["date"], df_series["NDVI"], marker="o", color="#2e8b57", label="mean NDVI in the ROI")
ax.axhline(df_series["NDVI"].min(), color="#a6611a", linestyle="--", label="min")
ax.axhline(df_series["NDVI"].max(), color="#1a9641", linestyle="--", label="max")
ax.set_title(f"NDVI in the ROI during {START[:4]}")
ax.set_xlabel("Date")
ax.set_ylabel("NDVI")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

---

## 4.5 - Máscaras por umbral

Una máscara convierte un índice continuo en una imagen de **sí / no**:

- **Vegetación**: `NDVI > 0.3`
- **Agua**: `NDWI > 0.1`

`gt()` devuelve 1 donde la condición es verdadera y 0 donde es falsa. Luego:

- `selfMask()` oculta los ceros, de modo que solo la clase se dibuja sobre la imagen RGB;
- `updateMask(mask)` aplica la máscara a **otra** imagen - por ejemplo, para mostrar los colores RGB
  solo donde hay vegetación.

Estos umbrales son un punto de partida simple. Varían según el sensor, la fecha y el lugar - siempre
verifíquelos visualmente.

In [ ]:
# Máscaras binarias (1 = clase, 0 = no clase)
veg_mask = scene.select("NDVI").gt(0.3).rename("vegetation")
water_mask = scene.select("NDWI").gt(0.1).rename("water")

Map_mask = geemap.Map()
Map_mask.centerObject(AOI, 10)

# Composición RGB de fondo
Map_mask.addLayer(scene, vis_rgb, "RGB (natural colour)")

# selfMask: dibujar solo los píxeles donde la máscara es 1
Map_mask.addLayer(veg_mask.selfMask(), {"palette": ["#1a9641"]}, "Vegetation (NDVI > 0.3)")
Map_mask.addLayer(water_mask.selfMask(), {"palette": ["#2c7bb6"]}, "Water (NDWI > 0.1)")

# updateMask: la imagen RGB solo donde hay vegetación
Map_mask.addLayer(scene.updateMask(veg_mask), vis_rgb, "RGB only on vegetation", False)

Map_mask.add_legend(title="Masks", legend_dict={"Vegetation": "#1a9641", "Water": "#2c7bb6"})
Map_mask

### ¿Cuánta área cubre cada máscara?

`ee.Image.pixelArea()` da el área de cada píxel en m2. Multiplicándola por la máscara (1 o
0) y sumando sobre el AOI se obtiene el área de cada clase.

In [ ]:
# Área de cada píxel (m2) por la máscara (1/0) -> área de la clase en cada píxel
areas = (veg_mask.addBands(water_mask)
                 .multiply(ee.Image.pixelArea())
                 .reduceRegion(reducer=ee.Reducer.sum(), geometry=AOI, scale=30, maxPixels=1e10)
                 .getInfo())

# m2 -> km2
df_area = pd.DataFrame({"area (km2)": areas}).divide(1e6).round(1)
df_area

---

## 4.6 - Resumen: nuestro cubo de datos

| Paso | Qué hicimos | Herramienta de GEE |
|---|---|---|
| 1 | Área de estudio y colección Landsat 8 + 9 | `filterBounds`, `filterDate`, `filter`, `merge` |
| 2 | Eliminar los píxeles con nubes | `QA_PIXEL`, `bitwiseAnd`, `updateMask` |
| 3 | Unidades físicas | factor de escala `* 0.0000275 - 0.2` |
| 4 | Índices como nuevas bandas | `normalizedDifference`, `expression`, `addBands` |
| 5 | Variabilidad de los píxeles en una ROI | `reduceRegion`, reductores combinados, `geemap.zonal_stats` |
| 6 | Variabilidad en el tiempo | `min()`, `max()` de la colección |
| 7 | Clases simples | `gt()`, `selfMask()`, `updateMask()` |

La imagen final (o la colección completa) con bandas ópticas + índices es la entrada para el
siguiente paso: la **clasificación supervisada de imágenes**, en la que, en lugar de elegir un umbral a mano,
un algoritmo aprende las reglas a partir de muestras.

In [ ]:
# Las bandas de nuestra escena preparada: la entrada para la clasificación
scene.bandNames().getInfo()

---

## 4.7 - Ejercicio

1. **Tres ROI.** Dibuje una ROI sobre agua, una sobre cultivos y una sobre la ciudad. Ejecute las
   estadísticas zonales para cada una y compare los gráficos de barras. ¿Qué banda separa mejor las tres?
2. **¿Qué índice de vegetación?** Compare NDVI, MSAVI y EVI en un área de vegetación dispersa. ¿Cuál
   da los valores más bajos, y por qué (piense en el suelo)?
3. **Cambie los umbrales.** Pruebe `NDVI > 0.2` y `NDVI > 0.5`, y `NDWI > 0` y
   `NDWI > 0.3`. ¿Cómo cambia el área? ¿En qué lugar del mapa aparecen las diferencias?
4. **Otra estación.** Cambie las fechas para usar solo la época de lluvias (diciembre-marzo). ¿Qué
   pasa con el número de imágenes y con la amplitud del NDVI?

**Entregable**: el notebook con sus ROI, gráficos y un párrafo corto para
cada pregunta.

---

## Referencias

**Índices**

- Rouse, J.W. et al. (1974). Monitoring vegetation systems in the Great Plains with ERTS.
  *Third ERTS Symposium*, NASA SP-351, 309-317. (NDVI)
- Qi, J., Chehbouni, A., Huete, A.R., Kerr, Y.H., Sorooshian, S. (1994). A modified soil
  adjusted vegetation index. *Remote Sensing of Environment*, 48(2), 119-126. (MSAVI)
- Huete, A. et al. (2002). Overview of the radiometric and biophysical performance of the
  MODIS vegetation indices. *Remote Sensing of Environment*, 83(1-2), 195-213. (EVI)
- McFeeters, S.K. (1996). The use of the Normalized Difference Water Index (NDWI) in the
  delineation of open water features. *International Journal of Remote Sensing*, 17(7),
  1425-1432. (NDWI)
- Gao, B.-C. (1996). NDWI - A normalized difference water index for remote sensing of
  vegetation liquid water from space. *Remote Sensing of Environment*, 58(3), 257-266.
  (el índice de humedad conocido hoy como NDMI)

**Earth Engine y Landsat**

- Landsat 8 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2>
- Landsat 9 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2>
- Factores de escala de Landsat Colección 2: <https://www.usgs.gov/faqs/how-do-i-use-scale-factor-landsat-level-2-science-products>
- Reducción por regiones: <https://developers.google.com/earth-engine/guides/reducers_reduce_region>
- Máscaras: <https://developers.google.com/earth-engine/guides/image_relational>
- geemap: <https://geemap.org>

**Libro (acceso abierto)**

- Cardille, J.A., Crowley, M.A., Saah, D., Clinton, N.E. (eds.) (2023). *Cloud-Based Remote
  Sensing with Google Earth Engine: Fundamentals and Applications*. Springer.
  <https://www.eefabook.org/>